# Семинар 5. Решающее дерево: от кода к поведению модели

На лекции мы строили дерево по критерию неоднородности и сравнивали MDI с permutation importance. Теперь исследуем **работающий классификатор**: как он хранит узлы, выбирает пороги и что происходит с качеством, когда меняется сложность дерева.

Код алгоритма уже готов. Сегодня его **читаем, запускаем и исследуем**, а не пишем с нуля. Все данные синтетические и создаются в notebook.

**Маршрут занятия:** небольшая таблица → собственное дерево → сравнение со `sklearn` → шумные данные → настройки сложности → MDI и PI.

Основные вопросы: что хранится в обученной модели? Почему низкая ошибка на train не гарантирует хороших прогнозов? Чем отличаются два способа измерить важность признака?

## 1. Небольшая задача с известным правилом

Начнём с 30 точек на плоскости. Класс 1 находится в верхнем правом прямоугольнике, остальные точки имеют класс 0. **Какие два пороговых условия выделят класс 1?**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_circles
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

In [ ]:
def small_grid():
    """30 точек: положительный класс — правый верхний прямоугольник."""
    x1, x2 = np.meshgrid(np.arange(1, 7), np.arange(1, 6))
    X = np.column_stack([x1.ravel(), x2.ravel()]).astype(float)
    y = ((X[:, 0] >= 4) & (X[:, 1] >= 3)).astype(int)
    return X, y


X_grid, y_grid = small_grid()
print('Объектов:', len(y_grid), '| классов 0:', sum(y_grid == 0), '| классов 1:', sum(y_grid == 1))

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.scatter(X_grid[y_grid == 0, 0], X_grid[y_grid == 0, 1],
           marker='o', label='Класс 0', s=65)
ax.scatter(X_grid[y_grid == 1, 0], X_grid[y_grid == 1, 1],
           marker='s', label='Класс 1', s=65)
ax.set(xlabel='$x_1$', ylabel='$x_2$', title='Две группы на сетке')
ax.set_xticks(range(1, 7)); ax.set_yticks(range(1, 6))
ax.grid(alpha=0.2); ax.legend(); plt.show()

Класс 1 задаётся правилом $x_1\ge 4$ **и** $x_2\ge 3$. Дерево выразит его последовательными проверками $x_1\le3.5$ и, только в правой ветви, $x_2\le2.5$.

Здесь легко проверить корректность алгоритма. Позже усложним данные, добавив шум.

## 2. Готовый `MyDecisionTreeClassifier`

В лекции мы разобрали CART как последовательность пороговых разбиений. Теперь посмотрим на **готовую реализацию**. Она должна уметь найти лучшее условие в узле, рекурсивно построить дерево и применять сохранённые условия к новым объектам.

Класс целиком приведён ниже. Не нужно читать его построчно: сначала разберём, **как устроены узлы**, а затем обратимся к ключевым методам.

### Как нумеруются узлы

Дерево хранится в обычном словаре `self.tree`: **ключ** — номер узла `node_id`, **значение** — описание узла. Для адресации используем правило:

```text
             0
           /   \
          1     2
         / \   / \
        3   4 5   6
```

Для узла `i` левый потомок имеет номер `2*i + 1`, правый — `2*i + 2`. У листа потомков нет, поэтому некоторые номера могут отсутствовать в словаре.

**Не путайте:** `node_id` — адрес узла, а `depth` — число переходов от корня (у корня глубина 0).

### Какие данные хранит узел

- **Внутренний узел:** `[NON_LEAF, feature, threshold, n, gain]` — признак и порог, число обучающих объектов и выигрыш критерия.
- **Лист:** `[LEAF, class_index, proba, n]` — индекс класса большинства, доли всех классов и число объектов.

Первое число — **тип записи**, а не предсказанный класс: `NON_LEAF = 0`, `LEAF = 1`. Именно по этому признаку `predict` понимает, нужно ли переходить к потомку или возвращать прогноз.

In [ ]:
class MyDecisionTreeClassifier:
    """Учебное CART-подобное дерево для числовых признаков и произвольных меток классов.

    Пороговые разбиения выбираются жадно по Gini или энтропии.
    Нулевой выигрыш допускается, если существует допустимое разбиение.
    Узлы хранятся в словаре tree с индексами детей 2*i+1 и 2*i+2.
    """

    LEAF = 1
    NON_LEAF = 0

    def __init__(self, max_depth=5, min_samples_split=2,
                 min_samples_leaf=1, criterion='gini'):
        """Задать ограничения роста и критерий ('gini' либо 'entropy')."""
        if criterion not in ('gini', 'entropy'):
            raise ValueError("criterion должен быть 'gini' или 'entropy'")
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.criterion = criterion
        self.tree = {}

    def _gini(self, y):
        """Индекс Gini для целочисленных индексов классов текущего узла."""
        if len(y) == 0:
            return 0.0
        p = np.bincount(y) / len(y)
        return float(1 - np.sum(p ** 2))

    def _entropy(self, y):
        """Энтропия меток в битах; нулевые доли не участвуют в сумме."""
        if len(y) == 0:
            return 0.0
        p = np.bincount(y) / len(y)
        p = p[p > 0]
        return float(-np.sum(p * np.log2(p)))

    def _impurity(self, y):
        """Выбранный критерий неоднородности."""
        return self._gini(y) if self.criterion == 'gini' else self._entropy(y)

    def _split_impurity(self, y_left, y_right):
        """Взвешенная неоднородность после разбиения текущего узла."""
        n = len(y_left) + len(y_right)
        return (len(y_left) / n * self._impurity(y_left)
                + len(y_right) / n * self._impurity(y_right))

    def _best_split(self, X, y):
        """Вернуть (индекс признака, порог, выигрыш) лучшего допустимого split.

        Кандидаты — середины соседних уникальных значений каждого столбца.
        Если кандидатов нет, индекс признака и порог равны None.
        При равном выигрыше сохраняется первый найденный вариант.
        """
        current = self._impurity(y)
        best_feature, best_threshold = None, None
        best_gain = -np.inf

        for feature in range(X.shape[1]):
            values = np.unique(X[:, feature])
            thresholds = (values[:-1] + values[1:]) / 2
            for threshold in thresholds:
                left = X[:, feature] <= threshold
                right = ~left
                y_left, y_right = y[left], y[right]
                if (len(y_left) < self.min_samples_leaf or
                        len(y_right) < self.min_samples_leaf):
                    continue
                gain = current - self._split_impurity(y_left, y_right)
                if best_feature is None or gain > best_gain:
                    best_feature = feature
                    best_threshold = float(threshold)
                    best_gain = float(gain)

        if best_feature is None:
            return None, None, 0.0
        return best_feature, best_threshold, best_gain

    def _make_leaf(self, y, node_id):
        """Сохранить индекс самого частого класса и частоты всех классов."""
        counts = np.bincount(y, minlength=self.n_classes_)
        predicted_index = int(np.argmax(counts))
        self.tree[node_id] = [self.LEAF, predicted_index,
                              counts / len(y), len(y)]

    def _fit_node(self, X, y, node_id, depth):
        """Обучить текущий узел, затем рекурсивно построить допустимых детей."""
        if self.max_depth is not None and depth >= self.max_depth:
            self._make_leaf(y, node_id)
            return
        if len(y) < self.min_samples_split or self._impurity(y) == 0:
            self._make_leaf(y, node_id)
            return
        feature, threshold, gain = self._best_split(X, y)
        if feature is None:
            self._make_leaf(y, node_id)
            return

        left = X[:, feature] <= threshold
        right = ~left
        self.tree[node_id] = [self.NON_LEAF, feature, threshold,
                              len(y), gain]
        self._fit_node(X[left], y[left], 2 * node_id + 1, depth + 1)
        self._fit_node(X[right], y[right], 2 * node_id + 2, depth + 1)

    def fit(self, X, y):
        """Кодировать метки, вырастить дерево с корня и вернуть self."""
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        if (X.ndim != 2 or y.ndim != 1 or len(X) != len(y)
                or len(y) == 0 or X.shape[1] == 0):
            raise ValueError('Нужны непустые X (n, p) и y (n,)')
        if not np.isfinite(X).all():
            raise ValueError('Признаки должны быть числовыми и без NaN/inf')
        self.classes_, y_encoded = np.unique(y, return_inverse=True)
        self.n_classes_ = len(self.classes_)
        self.tree = {}
        self._fit_node(X, y_encoded, node_id=0, depth=0)
        return self

    def _reach_leaf(self, x, node_id=0):
        """Пройти от заданного узла к листу и вернуть его запись."""
        node = self.tree[node_id]
        if node[0] == self.LEAF:
            return node
        _, feature, threshold, *_ = node
        if x[feature] <= threshold:
            return self._reach_leaf(x, 2 * node_id + 1)
        return self._reach_leaf(x, 2 * node_id + 2)

    def predict_single(self, x):
        """Вернуть исходную метку класса для одного объекта."""
        leaf = self._reach_leaf(np.asarray(x, dtype=float))
        return self.classes_[leaf[1]]

    def predict(self, X):
        """Метки классов для строк X (без переобучения)."""
        X = np.asarray(X, dtype=float)
        return np.asarray([self.predict_single(x) for x in X])

    def predict_proba(self, X):
        """Доли обучающих классов в достигнутых листьях, в порядке classes_."""
        X = np.asarray(X, dtype=float)
        return np.array([self._reach_leaf(x)[2] for x in X])

### Что происходит при `fit`?

`fit` один раз превращает произвольные метки классов в индексы `0, 1, …, K−1`. Затем `_fit_node` вызывает `_best_split`, сохраняет узел и при необходимости запускается для двух дочерних групп.

**Вопрос:** зачем в `fit` нужна пара `np.unique(..., return_inverse=True)`, если класс можно предсказывать по исходным меткам?

In [ ]:
labels = np.array(['кот', 'пёс', 'кот', 'птица'])
classes, indices = np.unique(labels, return_inverse=True)
print('Исходные классы:', classes)
print('Индексы в алгоритме:', indices)
print('Обратное преобразование:', classes[indices])

### Как `np.bincount` считает классы

`np.unique(..., return_inverse=True)` уже превратил метки в индексы `0, …, K−1`. `np.bincount` получает эти **целочисленные индексы**, а не текстовые названия. Посмотрим, почему в листьях нужен аргумент `minlength=K`.

In [ ]:
counts = np.bincount(indices, minlength=len(classes))
print('Число объектов каждого класса:', dict(zip(classes.tolist(), counts.tolist())))

# В этом листе нет объектов с индексом класса 2.
leaf_indices = np.array([0, 0, 1])
print('Без minlength:', np.bincount(leaf_indices))
print('С minlength:', np.bincount(leaf_indices, minlength=len(classes)))

В листе по-прежнему **три возможных класса**, даже если третий там не встретился. `minlength=K` дополняет вектор счётчиков нулём: длина `predict_proba` и порядок классов остаются одинаковыми для всех листьев.

Теперь видно, зачем `fit` сохраняет `classes_`: методы `_gini`, `_entropy` и `_make_leaf` считают частоты **индексов** классов, а `predict` возвращает исходные метки. Перейдём к выбору порогов.

### Как дерево выбирает порог?

У числового признака достаточно проверять середины **соседних уникальных** значений. Для каждого порога разделяем текущую группу на две и сравниваем взвешенный критерий.

Посмотрим на кандидатов первого признака нашей сетки.

In [ ]:
values = np.unique(X_grid[:, 0])
thresholds = (values[:-1] + values[1:]) / 2
print('Уникальные x1:', values)
print('Пороги-кандидаты:', thresholds)

**Какой из кандидатов создаст чистую левую группу, оставив положительные объекты справа?** Теперь проверим жадный выбор алгоритма.

In [ ]:
grid_tree = MyDecisionTreeClassifier(max_depth=3).fit(X_grid, y_grid)
root = grid_tree.tree[0]
print('Корень:', root)
print(f'Выбрано: x{root[1] + 1} <= {root[2]:g}')

На первом шаге выбрано `x1 ≤ 3.5`: слева остаются 15 объектов класса 0. Следующий полезный split находится уже в правой части. **Важно:** при рекурсии перебираются только объекты текущего узла.

### Как выглядит обученная структура?

Внутренний узел хранит правило перехода, а лист — прогноз. Распечатаем несколько записей, **не весь словарь целиком**.

In [ ]:
rows = []
for node_id in sorted(grid_tree.tree):
    node = grid_tree.tree[node_id]
    if node[0] == grid_tree.LEAF:
        description = f'лист: класс {grid_tree.classes_[node[1]]}'
        n = node[3]
    else:
        description = f'x{node[1] + 1} <= {node[2]:g}'
        n = node[3]
    rows.append({'узел': node_id, 'тип / правило': description, 'n': n})
pd.DataFrame(rows)

У корня номер `0`. Его дети — `1` и `2`; дети узла `2` — `5` и `6`. Так дерево хранится в словаре без отдельных объектов `Node`.

**Вопрос:** что происходит при `predict`? Новые пороги уже не ищутся: `_reach_leaf` проходит по сохранённым условиям.

In [ ]:
new_points = np.array([[5., 4.], [2., 4.], [5., 1.]])
print('Прогноз:', grid_tree.predict(new_points))
print('Доли классов в листьях:')
print(grid_tree.predict_proba(new_points))

На простой сетке дерево находит чистые листья, поэтому вероятности получаются равными 0 или 1. На данных с шумом доли обучающих классов в листьях уже не обязательно будут крайними. Это ещё не гарантирует калибровку вероятностей на новых объектах.

## 3. Схема дерева и области решений

Первый рисунок покажет сами условия и листья. Второй — области плоскости, в которых дерево возвращает один и тот же класс. Сопоставьте вертикальное условие в корне с границей на плоскости.

In [ ]:
def plot_custom_tree(model, *, figsize=(9.2, 4)):
    """Нарисовать небольшое учебное дерево из словаря model.tree."""
    coords = {}
    leaf_order = [0]

    def place(node_id, depth):
        node = model.tree[node_id]
        if node[0] == model.LEAF:
            x = leaf_order[0]
            leaf_order[0] += 1
        else:
            x_left = place(2 * node_id + 1, depth + 1)
            x_right = place(2 * node_id + 2, depth + 1)
            x = (x_left + x_right) / 2
        coords[node_id] = (x, -depth)
        return x

    place(0, 0)
    fig, ax = plt.subplots(figsize=figsize)
    for node_id, node in model.tree.items():
        x, y = coords[node_id]
        if node[0] != model.LEAF:
            for child_id in (2 * node_id + 1, 2 * node_id + 2):
                cx, cy = coords[child_id]
                ax.plot([x, cx], [y - 0.06, cy + 0.06], color='0.55')
            label = f'x{node[1]+1} <= {node[2]:.2f}\nn={node[3]}'
        else:
            label = f'Класс {model.classes_[node[1]]}\nn={node[3]}'
        ax.text(x, y, label, ha='center', va='center', fontsize=10,
                bbox=dict(boxstyle='round,pad=0.5', fc='0.94', ec='0.55'))
    ax.set(xlim=(-0.75, max(2, leaf_order[0] - 0.2)),
           ylim=(min(y for x, y in coords.values()) - 0.6, 0.65))
    ax.axis('off')
    return fig, ax

In [ ]:
fig, ax = plot_custom_tree(grid_tree)
ax.set_title('Собственное дерево на сетке')
plt.show()

Внутренние узлы содержат пороги; листья — класс и число наблюдений. Остаётся увидеть те же правила как границы классификации.

In [ ]:
def plot_boundary(model, X, y, *, ax=None, title=None, step=0.09):
    """Показать границу бинарного дерева на двух числовых признаках."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 4.4))
    x_lo, x_hi = X[:, 0].min() - 0.35, X[:, 0].max() + 0.35
    y_lo, y_hi = X[:, 1].min() - 0.35, X[:, 1].max() + 0.35
    xx, yy = np.meshgrid(np.arange(x_lo, x_hi, step),
                         np.arange(y_lo, y_hi, step))
    points = np.column_stack([xx.ravel(), yy.ravel()])
    predictions = model.predict(points).reshape(xx.shape)
    ax.contourf(xx, yy, predictions, levels=[-0.5, 0.5, 1.5],
                cmap='Pastel1', alpha=0.75)
    for cls, marker in [(0, 'o'), (1, 's')]:
        mask = y == cls
        ax.scatter(X[mask, 0], X[mask, 1], label=f'Класс {cls}',
                   marker=marker, s=28, edgecolors='0.25', linewidths=0.35)
    ax.set(xlabel='$x_1$', ylabel='$x_2$', xlim=(x_lo, x_hi), ylim=(y_lo, y_hi))
    ax.legend(fontsize=9)
    if title:
        ax.set_title(title)
    return ax

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.2))
plot_boundary(grid_tree, X_grid, y_grid, ax=ax,
              title='Области, построенные нашим деревом', step=0.08)
plt.show()

Горизонтальная граница существует только внутри правой области — именно так и работает условие во втором внутреннем узле. **Вопрос:** смогла бы одна линейная граница выделить такой же прямоугольник без дополнительных преобразований признаков?

## 4. Короткое сравнение со `sklearn`

Обучим `DecisionTreeClassifier` с такой же допустимой глубиной и покажем правила. Одинаковая идея CART не обязывает реализации строить идентичные деревья: важны правила разрешения равных выигрышей и остановки.

In [ ]:
library_grid = DecisionTreeClassifier(max_depth=3, random_state=42)
library_grid.fit(X_grid, y_grid)
print(export_text(library_grid, feature_names=['x1', 'x2']))
print('Совпадение прогнозов на сетке:',
      np.array_equal(grid_tree.predict(X_grid), library_grid.predict(X_grid)))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
plot_tree(library_grid, feature_names=['x1', 'x2'],
          class_names=['0', '1'], filled=True, rounded=True,
          fontsize=10, ax=ax)
plt.show()

Сверьте пороги и листья с нашим рисунком. `sklearn` хранит структуру иначе, но `fit` и `predict` решают ту же задачу: построить правило из данных и применить его к новым объектам.

## 5. Данные с шумом: проверяем сложность

На простой сетке дерево справляется почти идеально. Теперь сгенерируем две шумные окружности: класс зависит от положения на плоскости, но некоторые наблюдения перекрываются. **Будет ли глубокое дерево полезнее на новых объектах?**

In [ ]:
X_all, y_all = make_circles(
    n_samples=400, factor=0.45, noise=0.25, random_state=42
)
print('Данные:', X_all.shape, '| доли классов:', np.bincount(y_all))

Сначала отделим test. Из оставшейся dev-части выделим train и validation. Настройки дерева будем выбирать **только по validation**, не заглядывая в test.

In [ ]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X_all, y_all, test_size=0.20, stratify=y_all, random_state=21
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=21
)
print('Train:', len(y_train), '| validation:', len(y_valid), '| test:', len(y_test))

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1],
           s=27, marker='o', label='Класс 0', alpha=0.75)
ax.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1],
           s=27, marker='s', label='Класс 1', alpha=0.75)
ax.set(xlabel='$x_1$', ylabel='$x_2$', title='Train: шумные окружности')
ax.set_aspect('equal'); ax.legend(); plt.show()

Это синтетические признаки, а не временной ряд. Для них достаточно обычного случайного разбиения с сохранением долей классов. Никакое масштабирование для осевых порогов нам не требуется.

### Отправная точка

Для сравнения обучим классификатор, который всегда предсказывает наиболее частый класс train. Его accuracy помогает понять масштаб задачи.

In [ ]:
baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
print('Baseline accuracy на validation:',
      round(accuracy_score(y_valid, baseline.predict(X_valid)), 3))

### Собственное дерево и библиотечное

Сравним две реализации при `max_depth=3`. Их качество может совпадать, даже если некоторые выбранные правила отличаются.

In [ ]:
our_circles = MyDecisionTreeClassifier(max_depth=3).fit(X_train, y_train)
sklearn_circles = DecisionTreeClassifier(max_depth=3, random_state=42)
sklearn_circles.fit(X_train, y_train)

pd.DataFrame({
    'Модель': ['Наше дерево', 'scikit-learn'],
    'Train accuracy': [accuracy_score(y_train, our_circles.predict(X_train)),
                       accuracy_score(y_train, sklearn_circles.predict(X_train))],
    'Validation accuracy': [accuracy_score(y_valid, our_circles.predict(X_valid)),
                            accuracy_score(y_valid, sklearn_circles.predict(X_valid))],
}).round(3)

**Что сравниваем?** Не только числа accuracy, но и форму получившихся правил. Посмотрим на те же обучающие объекты и два дерева.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), constrained_layout=True)
plot_boundary(our_circles, X_train, y_train, ax=axes[0], title='Наше дерево, depth=3')
plot_boundary(sklearn_circles, X_train, y_train, ax=axes[1], title='sklearn, depth=3')
plt.show()

Здесь дерево строит прямоугольные области, постепенно приближая нелинейную структуру данных. Следующий вопрос — когда дальнейшее дробление начинает подстраиваться под шум.

## 6. Меняем `max_depth`

С ростом максимальной глубины дерево получает больше возможностей уменьшать **обучающую** ошибку. Посмотрим, что при этом происходит с validation accuracy и числом листьев.

In [ ]:
depths = [1, 2, 3, 4, 5, 6, 8, None]
depth_rows = []
for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    depth_rows.append({
        'Глубина': str(depth) if depth is not None else 'без ограничения',
        'Train accuracy': accuracy_score(y_train, tree.predict(X_train)),
        'Validation accuracy': accuracy_score(y_valid, tree.predict(X_valid)),
        'Листьев': tree.get_n_leaves(),
    })
depth_table = pd.DataFrame(depth_rows)
depth_table.round(3)

Числа удобнее читать вместе с графиком. **На какой глубине validation перестаёт улучшаться, хотя train ещё улучшается?**

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.3))
ax.plot(depth_table['Глубина'], depth_table['Train accuracy'],
        marker='o', label='Train')
ax.plot(depth_table['Глубина'], depth_table['Validation accuracy'],
        marker='s', label='Validation')
ax.set(xlabel='Максимальная разрешённая глубина', ylabel='Accuracy',
       title='Точность и сложность дерева', ylim=(0.4, 1.03))
ax.grid(alpha=0.2); ax.legend(); plt.show()

Глубина задаёт **верхний предел**, а не обязательное число уровней. Теперь сопоставим не только accuracy, но и геометрию: как выглядят границы неглубокого и почти неограниченного дерева?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
for ax, depth in zip(axes, [2, 4, None]):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_train, y_train)
    label = 'без ограничения' if depth is None else f'depth={depth}'
    plot_boundary(tree, X_train, y_train, ax=ax, title=label)
plt.show()

На глубоком дереве появляются маленькие прямоугольники вокруг отдельных точек. Это само по себе не доказывает переобучение: вывод делаем по сравнению качества на train и validation.

## 7. Меняем `min_samples_leaf`

Теперь ограничим минимальное число обучающих объектов **в каждом дочернем листе**. При одинаковой максимальной глубине сравним 1, 5 и 20. Что должно произойти с числом листьев?

In [ ]:
leaf_rows = []
for leaf_size in [1, 5, 20]:
    tree = DecisionTreeClassifier(
        max_depth=8, min_samples_leaf=leaf_size, random_state=42
    ).fit(X_train, y_train)
    leaf_rows.append({
        'min_samples_leaf': leaf_size,
        'Листьев': tree.get_n_leaves(),
        'Train accuracy': accuracy_score(y_train, tree.predict(X_train)),
        'Validation accuracy': accuracy_score(y_valid, tree.predict(X_valid)),
    })
pd.DataFrame(leaf_rows).round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
for ax, leaf_size in zip(axes, [1, 5, 20]):
    tree = DecisionTreeClassifier(
        max_depth=8, min_samples_leaf=leaf_size, random_state=42
    ).fit(X_train, y_train)
    plot_boundary(tree, X_train, y_train, ax=ax,
                  title=f'min_samples_leaf={leaf_size}')
plt.show()

Увеличение минимального размера листа делает дерево менее гибким. Это может помочь обобщению, но чрезмерное ограничение мешает учесть реальные закономерности. **Ищем компромисс по validation**, а не по одному красивому рисунку.

## 8. Подбираем сложность с `GridSearchCV`

Ранее мы **исследовали** отдельные значения глубины и размера листа на train/validation. Для систематического выбора настройки используем `GridSearchCV`: он сравнит все сочетания на фолдах **только обучающей части `X_train`**.

`X_valid` не участвует ни в подборе, ни в `fit` выбранной модели. Её сохраним для последующего сравнения важностей MDI/PI; итоговый `test` по-прежнему закрыт.

In [ ]:
param_grid = {
    'max_depth': [2, 4, 6, 8],
    'min_samples_leaf': [1, 5, 20],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    scoring='accuracy',
    cv=cv,
    refit=True,
)

**Что сделает `.fit()`?** Для каждой комбинации `max_depth` и `min_samples_leaf` он оценит accuracy на пяти фолдах. Всего **12 комбинаций × 5 фолдов = 60 обучений**. После выбора победителя `refit=True` обучит итоговое дерево заново на **всех `X_train`**.

In [ ]:
search.fit(X_train, y_train)
print('Проверено комбинаций:', len(search.cv_results_['params']))

### Читаем `cv_results_`

Одна строка — одна комбинация параметров. `mean_test_score` здесь означает **среднюю accuracy на CV-фолдах**, а не оценку на нашем отложенном `X_test`. Сравним среднее и разброс по фолдам.

In [ ]:
cv_results = pd.DataFrame(search.cv_results_)
cv_table = cv_results[[
    'param_max_depth', 'param_min_samples_leaf',
    'mean_test_score', 'std_test_score', 'rank_test_score',
]].copy()
cv_table.columns = ['max_depth', 'min_samples_leaf', 'CV accuracy', 'CV std', 'Место']
cv_table.sort_values(['Место', 'max_depth', 'min_samples_leaf']).round(3)

**Обратите внимание на ничьи.** Несколько комбинаций могут получить одинаковую среднюю CV-accuracy. В обычном `GridSearchCV` при равенстве лучшего score выбирается первая встреченная комбинация — это **не** автоматический выбор дерева с минимальным числом листьев.

### Что сохранил `GridSearchCV`?

`best_params_` — выбранные настройки, `best_score_` — соответствующая средняя CV-accuracy. В `best_estimator_` уже находится дерево, **переобученное на всём `X_train`**. Его и будем исследовать далее.

In [ ]:
model = search.best_estimator_
print('Выбранные параметры:', search.best_params_)
print('Средняя accuracy по CV:', round(search.best_score_, 3))
print('Accuracy на validation:', round(accuracy_score(y_valid, model.predict(X_valid)), 3))
print('Число листьев:', model.get_n_leaves())

Для MDI и PI ниже используем **один и тот же объект `model`**. Внутренние разбиения построены на train; PI оценим на validation.

Важно: раньше мы уже просматривали validation-графики при разных глубинах. Поэтому анализ PI здесь **разведочный**, а не независимая финальная оценка качества. Отложенный test не используется.

## 9. MDI: какие признаки дерево использовало при обучении?

`feature_importances_` суммирует взвешенные уменьшения неоднородности по узлам и нормирует их. Это характеристика построенных на train правил, а не результат проверки на validation.

In [ ]:
feature_names = ['x1', 'x2']
mdi = pd.Series(model.feature_importances_, index=feature_names, name='MDI')
mdi.sort_values(ascending=False).to_frame().round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 2.6))
mdi.sort_values().plot.barh(ax=ax)
ax.set(xlabel='Доля снижения обучающей неоднородности',
       title='MDI для выбранного дерева', xlim=(0, 1))
plt.show()

Сумма MDI в этом дереве равна единице. **Если удалить признак из данных, сохранится ли его важность?** По одной MDI ответить нельзя: потребуется переобучение новой модели.

## 10. PI: как изменится качество при перестановке?

Для уже обученного `model` перемешаем каждый столбец **validation-таблицы** по отдельности. Прогнозы пересчитаются, но сама модель останется прежней.

In [ ]:
pi_result = permutation_importance(
    model, X_valid, y_valid,
    scoring='accuracy', n_repeats=12, random_state=42
)
pi_pp = pd.Series(100 * pi_result.importances_mean,
                  index=feature_names, name='PI, п.п. accuracy')
pi_pp.sort_values(ascending=False).to_frame().round(2)

PI выражена в **процентных пунктах accuracy**: значение 10 означает, что после перестановки accuracy в среднем упала на 0.10. Эта шкала принципиально отличается от долей MDI.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 2.6))
pi_pp.sort_values().plot.barh(ax=ax)
ax.axvline(0, color='0.4', linewidth=0.8)
ax.set(xlabel='Среднее снижение accuracy, процентные пункты',
       title='Permutation importance на validation')
plt.show()

### Сопоставляем методы

Для двух признаков сравним порядок, **но не высоту столбиков разных шкал**. MDI говорит, какими разбиениями дерево уменьшало Gini на train. PI говорит, насколько фиксированное дерево потеряло в validation accuracy после перестановки признака.

In [ ]:
pd.DataFrame({
    'MDI (доля)': mdi,
    'PI (п.п. accuracy)': pi_pp,
}).round(2)

## 11. Что установили

**Алгоритм.** Дерево перебирает допустимые пороги, сохраняет выигравшее условие и рекурсивно продолжает работу в дочерних группах. После обучения `predict` только применяет сохранённые правила.

**Обобщение.** Чем глубже дерево, тем больше возможностей подогнать train. На примерах мы исследовали глубину и размер листа, а затем выбрали комбинацию параметров по кросс-валидации внутри train с `GridSearchCV`.

**Важность.** MDI относится к построенным разбиениям, а PI — к качеству фиксированной модели при перестановках на оценочных данных. Ни та ни другая мера сама по себе не является причинным эффектом признака.

## Самостоятельная работа до конца недели

В этом notebook есть полный исходный `MyDecisionTreeClassifier`. Используйте его как отправную точку для самостоятельной разработки **`MyDecisionTreeRegressor`**. Готового регрессора или каркаса с пропущенными методами нет.

В отдельном условии `regression_tree_task.md` указаны требования, контрольный пример с шестью наблюдениями, сравнение с `sklearn.tree.DecisionTreeRegressor` и формат сдачи. Для работы достаточно ноутбука с вашим кодом и коротким выводом.